# BirdCLEF 2026 — Convert Perch v2_cpu/1 to ONNX (GCP VM)
## Run ONCE on GCP. Output uploaded as `birdclef-2026-perch-onnx`.

### Why ONNX?
- TF SavedModel on Kaggle is unreliable: `perch_v2/2` uses XLA module v10 (crashes);
  `perch_v2_cpu/1` works but TF itself is heavy and hard to version-pin.
- ONNX + `onnxruntime-cpu` is lightweight, deterministic, and works on every Kaggle runtime.
- The 0.944 public LB notebook uses exactly this approach (`Perch-onnx-for-birdclef+2026` dataset).

### Quick start:
```bash
export PERCH_MODEL=~/models/perch_v2_cpu/1   # path to saved_model dir
export BIRDCLEF_OUT=~/birdclef-output
```
Run all cells. Upload `birdclef-2026-perch-onnx` to Kaggle, then use it in inference v23/v24.

In [ ]:
# === CELL 1: INSTALL (run this cell, then RESTART KERNEL, then run Cell 2+) ===
# DO NOT reinstall tensorflow — Kaggle has a working version already.
# Only install tf2onnx and pin onnx to 1.15.0 (onnx>=1.16 breaks tf2onnx's node builder).
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'tf2onnx==1.16.1',
     'onnxruntime>=1.18',
     'onnx==1.15.0'],
    check=False,
)
print('\nDone. RESTART THE KERNEL now, then run Cell 2 onwards.')
print('(Kernel restart is required so the pinned onnx 1.15.0 is actually loaded)')

In [ ]:
# === CELL 2: IMPORTS (run AFTER restarting kernel) ===
import subprocess, sys, os
import numpy as np
import tensorflow as tf
import tf2onnx
import onnxruntime as ort
import onnx
from pathlib import Path

tf.get_logger().setLevel('ERROR')
tf.config.set_visible_devices([], 'GPU')
tf.config.optimizer.set_jit(False)

print(f'TF         : {tf.__version__}')
print(f'tf2onnx    : {tf2onnx.__version__}')
print(f'onnx       : {onnx.__version__}  (need 1.15.x)')
print(f'onnxruntime: {ort.__version__}')
assert onnx.__version__.startswith('1.15'), (
    f'onnx is {onnx.__version__} — restart kernel so 1.15.0 takes effect'
)

In [ ]:
# === CELL 3: PATHS ===
_perch_env = os.environ.get('PERCH_MODEL', '')
_out_root  = os.environ.get('BIRDCLEF_OUT', '/kaggle/working' if os.path.isdir('/kaggle/working') else '~/birdclef-output')

# Find the SavedModel directory
_perch_candidates = [
    _perch_env,
    # Kaggle — model added via Add Model → google/bird-vocalization-classifier
    '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1',
    '/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1',
    # GCP / local
    os.path.expanduser('~/models/perch_v2_cpu/1'),
    os.path.expanduser('~/models/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'),
]
PERCH_MODEL_PATH = None
for _c in _perch_candidates:
    if not _c:
        continue
    _p = pathlib.Path(_c)
    if _p.exists() and ((_p / 'saved_model.pb').exists() or any(_p.glob('*.pb'))):
        PERCH_MODEL_PATH = str(_p)
        print(f'Found SavedModel: {PERCH_MODEL_PATH}')
        break

if PERCH_MODEL_PATH is None:
    raise RuntimeError(
        'perch_v2_cpu/1 not found.\n'
        'On Kaggle: Add Model -> google/bird-vocalization-classifier -> tensorflow2 -> perch_v2_cpu -> 1\n'
        'On GCP/local: kaggle models instances versions download '
        'google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1 -p ~/models/'
    )

ONNX_OUT_DIR = pathlib.Path(_out_root).expanduser() / 'perch_onnx'
ONNX_OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = ONNX_OUT_DIR / 'perch_v2_cpu.onnx'

print(f'Output ONNX: {ONNX_PATH}')


In [ ]:
# === CELL 4: CONVERT TO ONNX (Python API) ===
# CLI tf2onnx hits a ValueError in onnx.helper.make_attribute with onnx>=1.16.
# Calling from_saved_model() directly bypasses that code path.
# opset 13 is safest for Perch — all onnxruntime >=1.14 support it.

print(f'Converting {PERCH_MODEL_PATH} -> {ONNX_PATH} ...')
print(f'  tf2onnx  : {tf2onnx.__version__}')
print(f'  onnx     : {onnx.__version__}')

_success = False
for _opset in [13, 12, 11]:
    try:
        model_proto, _ = tf2onnx.convert.from_saved_model(
            PERCH_MODEL_PATH,
            opset=_opset,
            output_path=str(ONNX_PATH),
            tag='serve',
            signature_def='serving_default',
        )
        print(f'\n✅ ONNX saved (opset={_opset}): {ONNX_PATH}  ({ONNX_PATH.stat().st_size / 1e6:.1f} MB)')
        _success = True
        break
    except Exception as _e:
        print(f'  opset={_opset} failed: {_e}')

if not _success:
    raise RuntimeError(
        'tf2onnx.convert.from_saved_model failed for all opsets.\n'
        'Ensure onnx==1.15.0 and tf2onnx==1.16.1 are installed (re-run Cell 1).'
    )

In [ ]:
# === CELL 5: INSPECT OUTPUT NAMES ===
# We need to know the exact output tensor name for the 1536-d embedding.

sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

print('ONNX inputs:')
for inp in sess.get_inputs():
    print(f'  name={inp.name!r}  shape={inp.shape}  type={inp.type}')

print('\nONNX outputs:')
ONNX_EMB_KEY = None
for out in sess.get_outputs():
    print(f'  name={out.name!r}  shape={out.shape}  type={out.type}')
    # Find the 1536-dim output
    if out.shape and len(out.shape) >= 2 and out.shape[-1] == 1536:
        ONNX_EMB_KEY = out.name
        print(f'  => SELECTED as embedding output')

if ONNX_EMB_KEY is None:
    # Inspect all outputs to pick the right one
    print('\nWARNING: Could not auto-detect 1536-d output. Testing all outputs with a dummy forward pass...')
    _dummy = np.zeros((1, 160000), dtype=np.float32)
    _all_outs = sess.run(None, {sess.get_inputs()[0].name: _dummy})
    for i, (out, val) in enumerate(zip(sess.get_outputs(), _all_outs)):
        print(f'  output[{i}] name={out.name!r} shape={val.shape}')
        if val.ndim >= 2 and val.shape[-1] == 1536:
            ONNX_EMB_KEY = out.name
            print(f'  => SELECTED')

assert ONNX_EMB_KEY is not None, 'Could not find 1536-d embedding output. Check model conversion.'
print(f'\nONNX_EMB_KEY = {ONNX_EMB_KEY!r}')

# Save key alongside model for reference
import json
with open(ONNX_OUT_DIR / 'model_info.json', 'w') as f:
    inp_name = sess.get_inputs()[0].name
    json.dump({'input_name': inp_name, 'embedding_output_name': ONNX_EMB_KEY}, f, indent=2)
print(f'Saved model_info.json: input={inp_name!r}, embedding={ONNX_EMB_KEY!r}')

In [ ]:
# === CELL 6: VERIFY WITH SINE WAVE ===
PERCH_SR     = 32000
PERCH_TARGET = PERCH_SR * 5  # 160,000

_inp_name = sess.get_inputs()[0].name
_test     = np.sin(2 * np.pi * 440 * np.linspace(0, 5, PERCH_TARGET)).astype(np.float32)[None]
_all_outs = sess.run(None, {_inp_name: _test})

# Get the embedding output
_output_names = [o.name for o in sess.get_outputs()]
_emb = _all_outs[_output_names.index(ONNX_EMB_KEY)]
if _emb.ndim == 3:
    _emb = _emb.mean(axis=1)   # (1, T, 1536) -> (1, 1536)
_emb = _emb[0]  # (1536,)

assert _emb.shape == (1536,), f'Expected (1536,), got {_emb.shape}'
assert _emb.std() > 0.1, f'Embedding std={_emb.std():.4f} is near zero — model may be corrupt'

print(f'✅ Verification passed: shape={_emb.shape}, mean={_emb.mean():.4f}, std={_emb.std():.4f}')
del sess  # free memory

In [ ]:
# === CELL 7: UPLOAD TO KAGGLE AS birdclef-2026-perch-onnx ===
import json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')

meta = {
    'title': 'birdclef-2026-perch-onnx',
    'id': f'{KAGGLE_USERNAME}/birdclef-2026-perch-onnx',
    'licenses': [{'name': 'CC0-1.0'}],
    'notes': 'Perch v2_cpu/1 converted to ONNX (opset 15). Use with onnxruntime>=1.14.',
}
with open(ONNX_OUT_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

files = list(ONNX_OUT_DIR.glob('*'))
print(f'Files to upload: {[f.name for f in files]}')
os.system(f'kaggle datasets create -p {ONNX_OUT_DIR} --dir-mode zip')
print('\n✅ Upload done!')
print('Next: add chiragggg/birdclef-2026-perch-onnx as input to inference-v23 and inference-v24.')
print('File path on Kaggle: /kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx')